# Model 1: Classical ARIMA Model for Drug `N05C`

## Hyperparameter Selection Methodology:
ARIMA order candidates $(p,d,q)$ evaluated via AIC & Validation set RMSLE.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N05C'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N05C loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: ARIMA Order (p, d, q) Grid Search Code
from statsmodels.tsa.arima.model import ARIMA

order_candidates = [(0,1,1), (1,1,0), (1,1,1), (2,1,1), (1,1,2)]
best_order = None
best_val_rmsle = float('inf')

print("=== ARIMA Order Grid Search on 2018 Validation Set ===")
for order in order_candidates:
    try:
        m = ARIMA(np.log1p(train_series), order=order).fit()
        pred_log = m.forecast(steps=len(val_series))
        pred_val = np.clip(np.expm1(pred_log), 0, None)
        met = evaluate_metrics(val_series.values, pred_val)['RMSLE']
        print(f"  * Order {order} : Val RMSLE = {met:.6f} | AIC = {m.aic:.2f}")
        if met < best_val_rmsle:
            best_val_rmsle = met
            best_order = order
    except Exception as e:
        continue

print(f"Selected Optimal ARIMA Order (p*, d*, q*): {best_order}")


=== ARIMA Order Grid Search on 2018 Validation Set ===
  * Order (0, 1, 1) : Val RMSLE = 0.541815 | AIC = 2015.18


  * Order (1, 1, 0) : Val RMSLE = 0.617159 | AIC = 2591.15
  * Order (1, 1, 1) : Val RMSLE = 0.541920 | AIC = 2017.17


  * Order (2, 1, 1) : Val RMSLE = 0.542374 | AIC = 2018.90


  * Order (1, 1, 2) : Val RMSLE = 0.541837 | AIC = 2019.13
Selected Optimal ARIMA Order (p*, d*, q*): (0, 1, 1)


In [3]:
# Step 2: Fit Selected ARIMA Order on Train+Val & Forecast 2019 Test
m1_model = ARIMA(np.log1p(combined_series), order=best_order)
m1_fit = m1_model.fit()
print(m1_fit.summary())

pred_log = m1_fit.forecast(steps=len(test_series))
m1_test_pred = np.clip(np.expm1(pred_log), 0, None)
m1_test_pred.index = test_series.index

test_metrics = evaluate_metrics(test_series, m1_test_pred)
print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 1: ARIMA{best_order} ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_ARIMA': m1_test_pred.values}).to_csv('m1_arima_preds.csv', index=False)


                               SARIMAX Results                                
Dep. Variable:                   N05C   No. Observations:                 1825
Model:                 ARIMA(0, 1, 1)   Log Likelihood               -1294.989
Date:                Sat, 15 Aug 2026   AIC                           2593.977
Time:                        19:58:43   BIC                           2604.995
Sample:                    01-02-2014   HQIC                          2598.042
                         - 12-31-2018                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.9832      0.004   -220.944      0.000      -0.992      -0.974
sigma2         0.2418      0.007     36.683      0.000       0.229       0.255
Ljung-Box (L1) (Q):                   0.01   Jarque-